In [ ]:
import h5py
import numpy as np
import pandas as pd
from astropy.io import fits
from tqdm import tqdm
import os
import healpy as hp
from ligo.skymap.io.fits import read_sky_map
from ligo.skymap.moc import uniq2nest, uniq2pixarea
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import torch

In [ ]:
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

# gw_df = pd.read_csv(f"{_BASE}/ML+GW+KN/dataset/O5_sim_bns/injections_final.csv")
SIM_NAME = "LSST_KN_NSBH_AUG"
SIM_PATH = f"{_BASE}/SNANA/SNDATA_ROOT/SIM/{SIM_NAME}/"
Neg_path = f"{_BASE}/data/ELASTICC_TRAIN/"
output_path = f"{_BASE}/data/{SIM_NAME}/"
BAND_MAP = {                   # Mapping filters to channel indices
    'LSST-u': 0, 'LSST-g': 1, 'LSST-r': 2, 'LSST-i': 3, 'LSST-z': 4, 'LSST-Y': 5,
}
NEG_BAND_MAP = {               # Mapping filters to channel indices for negative samples
    'u': 0, 'g': 1, 'r': 2, 'i': 3, 'z': 4, 'Y': 5,
}
NUM_BANDS = 6
MAX_LC_LENGTH = 200  # Maximum length of light curves all band

In [ ]:
gw_df = pd.read_csv(f"{_BASE}/ML+GW+KN/dataset/O5_sim_nsbh_aug/injections_final.csv")
# process gw data
valid_index = []

for gw_idx, row in tqdm(gw_df.iterrows(), total=len(gw_df)):
    event_id = int(row['simulation_id'])
    sim_dir = os.path.join(SIM_PATH, f"{SIM_NAME}_{event_id}")
    try:
        with fits.open(os.path.join(sim_dir, f"{SIM_NAME}_{event_id}_HEAD.FITS")) as hdul:
            head_data = hdul[1].data
            if len(head_data) == 0:
                continue
    except Exception as e:  # simulation failed, path do not exists
        continue
    valid_index.append(gw_idx)

print(f"Total valid GW+Optical samples: {len(valid_index)}")
valid_gw_df = gw_df.loc[valid_index]
valid_gw_df.reset_index(drop=True, inplace=True)
valid_gw_df.head()

In [ ]:
valid_gw_df.to_csv(f"{_BASE}/data/LSST_KN_BNS_AUG/gw_catalog.csv", index=False)

In [ ]:
valid_gw_df = pd.read_csv(f"{_BASE}/data/LSST_KN_BNS/gw_catalog.csv")
valid_gw_df

In [ ]:
# find max_lc_length
max_lc_length = 0
min_lc_length = 1e3
min_id = -1
max_id = -1
nobs = np.array([])
for gw_idx, row in tqdm(valid_gw_df.iterrows(), total=len(valid_gw_df)):
    event_id = int(row['simulation_id'])
    sim_dir = os.path.join(SIM_PATH, f"{SIM_NAME}_{event_id}")
    with fits.open(os.path.join(sim_dir, f"{SIM_NAME}_{event_id}_HEAD.FITS")) as hdul:
        head_data = hdul[1].data
        nobs = np.concatenate((nobs, head_data['NOBS']))
        if len(head_data) == 0:
            continue
        if min(head_data['NOBS']) < min_lc_length:
            min_lc_length = min(head_data['NOBS'])
            min_id = event_id
        if max(head_data['NOBS']) > max_lc_length:
            max_lc_length = max(head_data['NOBS'])
            max_id = event_id
print(f"Light curve length statistics over {len(nobs)} light curves:")
print(f"Max LC length across all bands: {max_lc_length} for event {max_id}")
print(f"Min LC length across all bands: {min_lc_length} for event {min_id}")

In [ ]:
import matplotlib.pyplot as plt
plt.hist(nobs, bins=50, range=(0,100))
print("Number of light curves with less than 5 observations:", np.sum(nobs < 5))

In [ ]:
# Functions for parsing SNANA FITS files, and sampling MOC skymaps
def parse_snana_fits(event_id, sim_dir, sim_name="LSST_KN_BNS_AUG"):
    """
    Parses {event_id}_HEAD.fits and {event_id}_PHOT.fits.
    Extracts multiple light curve realizations for a single GW event.
    
    Args:
        event_id: String ID of the event.
        sim_dir: Directory containing FITS files.
        
    Returns:
        List of tuples: [(values, masks, times), ...]
        Returns empty list if files are missing.
    """
    if type(event_id) == str:
        event_id = int(float(event_id))
    head_path = os.path.join(sim_dir, f"{sim_name}_{event_id}",f"{sim_name}_{event_id}_HEAD.FITS")
    phot_path = os.path.join(sim_dir, f"{sim_name}_{event_id}",f"{sim_name}_{event_id}_PHOT.FITS")

    if not os.path.exists(head_path) or not os.path.exists(phot_path):
        print(f"Warning: FITS files not found for {event_id}")
        return []

    try:
        # open readme file and get MJD explode value
        with open(os.path.join(sim_dir, f"{sim_name}_{event_id}",f"{sim_name}_{event_id}.README")) as f:
            readme_lines = f.readlines()
            mjd_explode = readme_lines[27].split(":")[1].split()[0]
            mjd_explode = float(mjd_explode)
        # Open FITS files
        with fits.open(head_path) as hdul_head, fits.open(phot_path) as hdul_phot:
            # Usually data is in extension 1
            data_head = hdul_head[1].data
            data_phot = hdul_phot[1].data
            
            # Use columns directly (Astropy FITS columns are case-insensitive usually)
            # HEAD columns
            ptrobs_min = data_head['PTROBS_MIN']
            ptrobs_max = data_head['PTROBS_MAX']
            
            # PHOT columns
            mjd_all = data_phot['MJD']
            flux_all = data_phot['FLUXCAL']
            fluxerr_all = data_phot['FLUXCALERR'] # Optional usage
            flt_all = data_phot['BAND'] # Filters

            extracted_lcs = []
            
            # Iterate over each realization in HEAD
            for i in range(len(data_head)):
                # SNANA uses 1-based indexing for pointers, Python uses 0-based
                # Start index: value - 1
                # End index: value (exclusive in python slicing)
                start_idx = ptrobs_min[i] - 1
                end_idx = ptrobs_max[i]
                nobs = data_head['NOBS'][i]
                if nobs < 5:
                    # print(f"Warning: Light curve for event {event_id} realization {i} has less than 5 observations. Skipping.")
                    continue  # Skip light curves with less than 5 observations

                # get coordinates
                ra = data_head['RA'][i]
                dec = data_head['DEC'][i]
                coordinates = np.array([ra, dec], dtype=np.float32)
                
                # Slicing the PHOT data
                lc_mjd = mjd_all[start_idx : end_idx]
                lc_flux = flux_all[start_idx : end_idx]
                lc_fluxerr = fluxerr_all[start_idx : end_idx]
                lc_flt = flt_all[start_idx : end_idx]

                # Normalization
                std = np.std(lc_flux)
                mean = np.mean(lc_flux)
                lc_flux = (lc_flux - mean) / (std + 1e-8)
                lc_fluxerr = lc_fluxerr / (std + 1e-8)
                
                # --- Format Conversion (to Tensor-ready numpy) ---
                val_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)    # Values matrix (flux)
                err_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)    # Errors matrix (flux errors)
                mask_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)
                time_vec = np.zeros((MAX_LC_LENGTH,), dtype=np.float32)
                
                # 1. Time Normalization (Relative to BNS merger time)
                if len(lc_mjd) > 0:
                    rel_times = (lc_mjd - mjd_explode) / 100  # Scale down to manageable range[-0.3, 0.6]
                else:
                    continue # Skip empty light curves

                # 2. Fill Matrices
                # Truncate if longer than MAX_LC_LENGTH
                seq_len = min(len(lc_mjd), MAX_LC_LENGTH)
                if len(lc_mjd) > MAX_LC_LENGTH:
                    print(f"Warning: Light curve for event {event_id} exceeds MAX_LC_LENGTH. Truncating.")
                    # Keep the MAX_LC_LENGTH points with smallest absolute rel_times
                    sorted_indices = np.argsort(np.abs(rel_times))[:MAX_LC_LENGTH]
                    sorted_indices = np.sort(sorted_indices)  # Sort back to chronological order
                    lc_mjd = lc_mjd[sorted_indices]
                    lc_flux = lc_flux[sorted_indices]
                    lc_fluxerr = lc_fluxerr[sorted_indices]
                    lc_flt = lc_flt[sorted_indices]
                    rel_times = rel_times[sorted_indices]
                
                for t in range(seq_len):
                    band_char = lc_flt[t].strip() # Remove whitespace
                    if band_char in BAND_MAP:
                        b_idx = BAND_MAP[band_char]
                        
                        val_mat[t, b_idx] = lc_flux[t]
                        err_mat[t, b_idx] = lc_fluxerr[t]
                        mask_mat[t, b_idx] = 1.0
                        time_vec[t] = rel_times[t]
                
                extracted_lcs.append((val_mat, err_mat, mask_mat, time_vec, coordinates))
                
            return extracted_lcs

    except Exception as e:
        print(f"Error processing FITS for {event_id}: {e}")
        return []

def sample_moc_skymap(map_file):
    """
    Convert UNIQ to sky coordinates and calculte pixel areas.
    Note: Do not include DISTNORM in the output. For inf values in DISTMU, relace with distmean and diststd from metadata.
    """
    
    # read moc skymap and metadata
    moc_map = read_sky_map(map_file, moc=True, distances=True)
    _, meta = read_sky_map(map_file, nest=True)

    # extract distance meta info
    dist_mean = meta.get('distmean', None)
    dist_std = meta.get('diststd', None)

    uniq = moc_map['UNIQ']
    probdensity = moc_map['PROBDENSITY']
    distmu = moc_map['DISTMU']
    distsigma = moc_map['DISTSIGMA']
    # distnorm = moc_map['DISTNORM']  # not used

    # 1) UNIQ -> order, ipix, nside
    order, ipix = uniq2nest(uniq)

    # 2) caculate pixel area
    dA = uniq2pixarea(uniq)
    # 3) calculate pixel probability
    dP = probdensity * dA
    # 4) calculate theta, phi
    xs  = np.zeros_like(ipix, dtype=np.float32)
    ys = np.zeros_like(ipix, dtype=np.float32)
    zs = np.zeros_like(ipix, dtype=np.float32)
    for k in np.unique(order):
        m = (order == k)
        this_ipix  = ipix[m]
        this_nside = 2 ** k
        theta, phi = hp.pix2ang(this_nside, this_ipix, nest=True)
        # ras[m]  = np.degrees(phi)
        # decs[m] = 90.0 - np.degrees(theta)
        xs[m] = np.sin(theta) * np.cos(phi)   # dec,[0, pi]
        ys[m] = np.sin(theta) * np.sin(phi)   # ra,[0, 2pi]
        zs[m] = np.cos(theta)
    
    # 5) return torch tensors
    gw_mocmap = torch.tensor(np.vstack([xs, ys, zs, dA, 100 * dP, distmu, distsigma]), dtype=torch.float32)   # [7, N_pixels], no distnorm

    # 6) process unnormal distance values
    inf_dist_mu = torch.where(torch.isinf(gw_mocmap[5]))[0]
    gw_mocmap[5, inf_dist_mu] = dist_mean  # set inf to mean value
    gw_mocmap[6, inf_dist_mu] = dist_std   # set inf to std value
    gw_mocmap[5,:] = gw_mocmap[5,:] / 1000.0  # scale down
    gw_mocmap[6,:] = gw_mocmap[6,:] / 1000.0 # scale down

    return gw_mocmap  # [7, N_pixels]

In [ ]:
gw_moc = sample_moc_skymap(f"{_BASE}/data/bns_skymap/16.fits")
gw_moc

In [ ]:
for event_id in tqdm(valid_gw_df['simulation_id'], total=len(valid_gw_df)):
    gw_moc = sample_moc_skymap(f"{_BASE}/data/bns_skymap/{event_id}.fits")
    if torch.isnan(gw_moc).any() or torch.isinf(gw_moc).any():
        print(f"NaN or Inf detected in MOC skymap for event {event_id}.")

In [ ]:
lcs = parse_snana_fits(event_id=0,
                       sim_dir=f"{_BASE}/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS_AUG",
                       sim_name="LSST_KN_BNS_AUG")
len(lcs)

In [ ]:
# Function to create relational dataset
def create_relational_dataset(
    gw_catalog_path, 
    fits_dir, 
    output_h5_path
):
    """
    Main function to process all data and save to HDF5.
    """
    # 1. Load GW Catalog
    print(f"Loading GW Catalog from {gw_catalog_path}...")
    # Assuming CSV has columns: event_id, m1, m2, ..., skymap_path
    gw_df = pd.read_csv(gw_catalog_path)

    # rescale and normalize gw parameters
    # gw_df['mjd_time'] = (gw_df['mjd_time'] - 61000) / 100.0
    gw_df['inclination'] = np.cos(gw_df['inclination'])
    # scale down distance parameters
    gw_df['distmean'] = gw_df['distmean'] / 1000.0
    gw_df['diststd'] = gw_df['diststd'] / 1000.0

    n_unique = len(gw_df)
    
    # 2. Initialize HDF5 File
    with h5py.File(output_h5_path, 'w') as f:
        # --- Group A: Unique GW Events ---
        grp_gw = f.create_group('events/gw_data')
        
        # Pre-allocate GW datasets (we know exact size N_unique)
        ds_gw_scalars = grp_gw.create_dataset('scalars', (n_unique, 7), dtype='f4')
        ds_gw_skymaps = grp_gw.create_dataset('skymaps', (n_unique, 7, 19200), dtype='f4') # 7 channels after cleaning
        # Store IDs as fixed-length ASCII strings
        dt_str = h5py.special_dtype(vlen=str) 
        ds_gw_ids = grp_gw.create_dataset('ids', (n_unique,), dtype=dt_str)
        
        # --- Group B: All Optical Data ---
        # We don't know total optical count yet, so we use resizable datasets (chunked)
        grp_opt = f.create_group('events/optical_data')
        
        chunk_size = 1024
        ds_opt_vals = grp_opt.create_dataset('values', (0, MAX_LC_LENGTH, NUM_BANDS), 
                                             maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_errs = grp_opt.create_dataset('errors', (0, MAX_LC_LENGTH, NUM_BANDS),
                                             maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_masks = grp_opt.create_dataset('masks', (0, MAX_LC_LENGTH, NUM_BANDS), 
                                              maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_times = grp_opt.create_dataset('times', (0, MAX_LC_LENGTH), 
                                              maxshape=(None, MAX_LC_LENGTH), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH))
        ds_opt_coordinates = grp_opt.create_dataset('coordinates', (0, 2),
                                                   maxshape=(None, 2), dtype='f4', chunks=(chunk_size, 2))
        
        # Parent Index Mapping (The Relation)
        ds_parent_idx = grp_opt.create_dataset('parent_gw_idx', (0,), maxshape=(None,), dtype='i4', chunks=(chunk_size,))
        
        # --- Processing Loop ---
        print("Starting processing loop...")
        total_optical_count = 0
        
        # Buffer for optical data to reduce HDF5 resize calls (optimization)
        opt_buffer_vals = []
        opt_buffer_errs = []
        opt_buffer_masks = []
        opt_buffer_times = []
        opt_buffer_p_idx = []
        opt_buffer_coordinates = []
        BUFFER_LIMIT = 5000 

        def flush_buffer():
            nonlocal total_optical_count, opt_buffer_vals, opt_buffer_errs, opt_buffer_masks, opt_buffer_times, opt_buffer_p_idx, opt_buffer_coordinates
            if len(opt_buffer_vals) == 0: return
            
            n_new = len(opt_buffer_vals)
            current_size = total_optical_count
            new_size = current_size + n_new
            
            # Resize datasets
            ds_opt_vals.resize(new_size, axis=0)
            ds_opt_errs.resize(new_size, axis=0)
            ds_opt_masks.resize(new_size, axis=0)
            ds_opt_times.resize(new_size, axis=0)
            ds_parent_idx.resize(new_size, axis=0)
            ds_opt_coordinates.resize(new_size, axis=0)
            
            # Write data
            ds_opt_vals[current_size:new_size] = np.array(opt_buffer_vals)
            ds_opt_errs[current_size:new_size] = np.array(opt_buffer_errs)
            ds_opt_masks[current_size:new_size] = np.array(opt_buffer_masks)
            ds_opt_times[current_size:new_size] = np.array(opt_buffer_times)
            ds_parent_idx[current_size:new_size] = np.array(opt_buffer_p_idx)
            ds_opt_coordinates[current_size:new_size] = np.array(opt_buffer_coordinates)
            
            total_optical_count += n_new
            
            # Clear buffer
            opt_buffer_vals = []
            opt_buffer_errs = []
            opt_buffer_masks = []
            opt_buffer_times = []
            opt_buffer_p_idx = []
            opt_buffer_coordinates = []

        # Iterate over unique GW events
        for gw_idx, row in tqdm(gw_df.iterrows(), total=n_unique):
            # if gw_idx > 1:
            #     break
            event_id = int(row['simulation_id'])
            # print(f"Processing GW Event {event_id} ({gw_idx+1}/{n_unique})...")
            
            # 1. Process & Save GW Data
            # Scalars (Columns m1...param14)
            # Adjust columns based on your CSV
            gw_params_name = ['mass1_detector', 'mass2_detector', 'spin1z', 'spin2z', 'inclination', 'distmean', 'diststd']
            scalars = row[gw_params_name].values.astype(np.float32)

            # rescale parameters
            scalars

            ds_gw_scalars[gw_idx] = scalars
            ds_gw_ids[gw_idx] = str(event_id)
            
            # Skymap
            # Apply robust preprocessing (Returns Tensor [6, 19200])
            gw_mocmap = sample_moc_skymap(f"{_BASE}/data/bns_skymap/{event_id}.fits")
            ds_gw_skymaps[gw_idx] = gw_mocmap.numpy() # Convert back to numpy for HDF5
            
            # 2. Process Optical Data
            # Extract light curves from SNANA FITS
            lcs = parse_snana_fits(event_id, sim_dir=fits_dir)
            
            # Add to buffer
            for (vals, errs, masks, times, coordinates) in lcs:
                opt_buffer_vals.append(vals)
                opt_buffer_errs.append(errs)
                opt_buffer_masks.append(masks)
                opt_buffer_times.append(times)
                opt_buffer_p_idx.append(gw_idx) # Link to parent GW index
                opt_buffer_coordinates.append(coordinates)
            
            # Flush if buffer is full
            if len(opt_buffer_vals) >= BUFFER_LIMIT:
                flush_buffer()
        
        # Final flush
        flush_buffer()
        
        # Save metadata
        f.attrs['n_unique_gw'] = n_unique
        f.attrs['n_total_optical'] = total_optical_count
        print(f"\nProcessing Complete.")
        print(f"Unique GW Events: {n_unique}")
        print(f"Total Light Curves: {total_optical_count}")
        print(f"Saved to: {output_h5_path}")

In [ ]:
create_relational_dataset(
    gw_catalog_path=f"{_BASE}/data/LSST_KN_BNS/gw_catalog.csv",
    fits_dir=f"{_BASE}/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS",
    output_h5_path=f"{_BASE}/data/LSST_KN_BNS/combined_dataset.h5"
)

## Create Dataset with Negative GW Events

**Goal**: Include BNS events without detectable kilonova (KN) as "negative GW" events.

- **Positive GW**: BNS events with valid optical data (has_kn=1)
- **Negative GW**: BNS events with skymaps but NO optical data (has_kn=0)

These negative GW events represent real GW detections that should NOT match any optical transient.
During training, pairing a negative GW with a random optical sample should predict "no match" (cls_label=0).

In [ ]:
# Identify positive (with KN) and negative (without KN) GW events
# Load full BNS catalog
full_gw_df = pd.read_csv(f"{_BASE}/ML+GW+KN/dataset/O5_sim_nsbh_aug/injections_final.csv")
SKYMAP_DIR = f"{_BASE}/data/nsbh_skymap"

pos_indices = []  # Events with optical data
neg_indices = []  # Events without optical data but with skymaps

for gw_idx, row in tqdm(full_gw_df.iterrows(), total=len(full_gw_df), desc="Scanning BNS events"):
    event_id = int(row['simulation_id'])
    skymap_path = os.path.join(SKYMAP_DIR, f"{event_id}.fits")
    
    # Check if skymap exists (required for both positive and negative)
    if not os.path.exists(skymap_path):
        continue
    
    # Check if optical data exists
    sim_dir = os.path.join(SIM_PATH, f"{SIM_NAME}_{event_id}")
    head_path = os.path.join(sim_dir, f"{SIM_NAME}_{event_id}_HEAD.FITS")
    
    has_optical = False
    try:
        with fits.open(head_path) as hdul:
            if len(hdul[1].data) > 0:
                has_optical = True
    except:
        pass
    
    if has_optical:
        pos_indices.append(gw_idx)
    else:
        neg_indices.append(gw_idx)

print(f"\nTotal BNS events in catalog: {len(full_gw_df)}")
print(f"Positive GW (with KN): {len(pos_indices)}")
print(f"Negative GW (no KN but has skymap): {len(neg_indices)}")

In [ ]:
def create_dataset_with_neg_gw(
    full_catalog_path,
    skymap_dir,
    fits_dir,
    output_h5_path,
    pos_indices,
    neg_indices,
    max_neg_gw=None
):
    """
    Create HDF5 dataset with both positive (has KN) and negative (no KN) GW events.
    
    Args:
        full_catalog_path: Path to full BNS catalog CSV
        skymap_dir: Directory containing skymap FITS files
        fits_dir: Directory containing SNANA simulation FITS files
        output_h5_path: Output HDF5 file path
        pos_indices: List of indices in catalog for positive GW events
        neg_indices: List of indices in catalog for negative GW events
        max_neg_gw: Optional limit on number of negative GW events
    """
    # Load catalog
    gw_df = pd.read_csv(full_catalog_path)
    
    # Preprocess GW parameters
    gw_df['inclination'] = np.cos(gw_df['inclination'])
    gw_df['distmean'] = gw_df['distmean'] / 1000.0
    gw_df['diststd'] = gw_df['diststd'] / 1000.0
    # swap mass1 and mass2 if mass1 < mass2
    mask = gw_df['mass1_detector'] < gw_df['mass2_detector']
    gw_df.loc[mask, ['mass1_detector', 'mass2_detector']] = gw_df.loc[mask, ['mass2_detector', 'mass1_detector']].values
    
    # Optionally limit negative GW count
    if max_neg_gw and len(neg_indices) > max_neg_gw:
        neg_indices = np.random.choice(neg_indices, max_neg_gw, replace=False).tolist()
    
    n_pos = len(pos_indices)
    n_neg = len(neg_indices)
    n_total_gw = n_pos + n_neg
    
    print(f"Creating dataset with:")
    print(f"  Positive GW (with KN): {n_pos}")
    print(f"  Negative GW (no KN): {n_neg}")
    print(f"  Total GW events: {n_total_gw}")
    
    with h5py.File(output_h5_path, 'w') as f:
        # --- GW Data Group ---
        grp_gw = f.create_group('events/gw_data')
        
        ds_gw_scalars = grp_gw.create_dataset('scalars', (n_total_gw, 7), dtype='f4')
        ds_gw_skymaps = grp_gw.create_dataset('skymaps', (n_total_gw, 7, 19200), dtype='f4')
        dt_str = h5py.special_dtype(vlen=str)
        ds_gw_ids = grp_gw.create_dataset('ids', (n_total_gw,), dtype=dt_str)
        ds_gw_has_kn = grp_gw.create_dataset('has_kn', (n_total_gw,), dtype='i4')  # NEW: 1=has KN, 0=no KN
        
        # --- Optical Data Group (only for positive GW) ---
        grp_opt = f.create_group('events/optical_data')
        
        chunk_size = 1024
        ds_opt_vals = grp_opt.create_dataset('values', (0, MAX_LC_LENGTH, NUM_BANDS), 
                                             maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), 
                                             dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_errs = grp_opt.create_dataset('errors', (0, MAX_LC_LENGTH, NUM_BANDS),
                                             maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), 
                                             dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_masks = grp_opt.create_dataset('masks', (0, MAX_LC_LENGTH, NUM_BANDS), 
                                              maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), 
                                              dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_times = grp_opt.create_dataset('times', (0, MAX_LC_LENGTH), 
                                              maxshape=(None, MAX_LC_LENGTH), 
                                              dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH))
        ds_opt_coordinates = grp_opt.create_dataset('coordinates', (0, 2),
                                                   maxshape=(None, 2), 
                                                   dtype='f4', chunks=(chunk_size, 2))
        ds_parent_idx = grp_opt.create_dataset('parent_gw_idx', (0,), 
                                               maxshape=(None,), dtype='i4', chunks=(chunk_size,))
        
        # Buffers for optical data
        total_optical_count = 0
        opt_buffer_vals = []
        opt_buffer_errs = []
        opt_buffer_masks = []
        opt_buffer_times = []
        opt_buffer_p_idx = []
        opt_buffer_coordinates = []
        BUFFER_LIMIT = 5000
        
        def flush_buffer():
            nonlocal total_optical_count, opt_buffer_vals, opt_buffer_errs, opt_buffer_masks
            nonlocal opt_buffer_times, opt_buffer_p_idx, opt_buffer_coordinates
            if len(opt_buffer_vals) == 0: 
                return
            
            n_new = len(opt_buffer_vals)
            current_size = total_optical_count
            new_size = current_size + n_new
            
            ds_opt_vals.resize(new_size, axis=0)
            ds_opt_errs.resize(new_size, axis=0)
            ds_opt_masks.resize(new_size, axis=0)
            ds_opt_times.resize(new_size, axis=0)
            ds_parent_idx.resize(new_size, axis=0)
            ds_opt_coordinates.resize(new_size, axis=0)
            
            ds_opt_vals[current_size:new_size] = np.array(opt_buffer_vals)
            ds_opt_errs[current_size:new_size] = np.array(opt_buffer_errs)
            ds_opt_masks[current_size:new_size] = np.array(opt_buffer_masks)
            ds_opt_times[current_size:new_size] = np.array(opt_buffer_times)
            ds_parent_idx[current_size:new_size] = np.array(opt_buffer_p_idx)
            ds_opt_coordinates[current_size:new_size] = np.array(opt_buffer_coordinates)
            
            total_optical_count += n_new
            
            opt_buffer_vals = []
            opt_buffer_errs = []
            opt_buffer_masks = []
            opt_buffer_times = []
            opt_buffer_p_idx = []
            opt_buffer_coordinates = []
        
        gw_params_name = ['mass1_detector', 'mass2_detector', 'spin1z', 'spin2z', 
                         'inclination', 'distmean', 'diststd']
        
        # --- Process POSITIVE GW events (indices 0 to n_pos-1) ---
        print("\nProcessing positive GW events (with KN)...")
        gw_idx_counter = 0
        for orig_idx in tqdm(pos_indices, desc="Positive GW"):
            row = gw_df.iloc[orig_idx]
            event_id = int(row['simulation_id'])
            
            # Save GW data
            ds_gw_scalars[gw_idx_counter] = row[gw_params_name].values.astype(np.float32)
            ds_gw_skymaps[gw_idx_counter] = sample_moc_skymap(f"{skymap_dir}/{event_id}.fits").numpy()
            ds_gw_ids[gw_idx_counter] = str(event_id)
            ds_gw_has_kn[gw_idx_counter] = 1  # Has KN
            
            # Save optical data
            lcs = parse_snana_fits(event_id, sim_dir=fits_dir, sim_name=SIM_NAME)
            for (vals, errs, masks, times, coordinates) in lcs:
                opt_buffer_vals.append(vals)
                opt_buffer_errs.append(errs)
                opt_buffer_masks.append(masks)
                opt_buffer_times.append(times)
                opt_buffer_p_idx.append(gw_idx_counter)
                opt_buffer_coordinates.append(coordinates)
            
            if len(opt_buffer_vals) >= BUFFER_LIMIT:
                flush_buffer()
            
            gw_idx_counter += 1
        
        # Flush remaining positive optical data
        flush_buffer()
        
        # --- Process NEGATIVE GW events (indices n_pos to n_total-1) ---
        print("\nProcessing negative GW events (no KN)...")
        for orig_idx in tqdm(neg_indices, desc="Negative GW"):
            row = gw_df.iloc[orig_idx]
            event_id = int(row['simulation_id'])
            
            # Save GW data only (no optical)
            ds_gw_scalars[gw_idx_counter] = row[gw_params_name].values.astype(np.float32)
            ds_gw_skymaps[gw_idx_counter] = sample_moc_skymap(f"{skymap_dir}/{event_id}.fits").numpy()
            ds_gw_ids[gw_idx_counter] = str(event_id)
            ds_gw_has_kn[gw_idx_counter] = 0  # No KN
            
            gw_idx_counter += 1
        
        # Save metadata
        f.attrs['n_pos_gw'] = n_pos
        f.attrs['n_neg_gw'] = n_neg
        f.attrs['n_total_gw'] = n_total_gw
        f.attrs['n_total_optical'] = total_optical_count
        
        print(f"\nProcessing Complete.")
        print(f"  Positive GW Events: {n_pos}")
        print(f"  Negative GW Events: {n_neg}")
        print(f"  Total GW Events: {n_total_gw}")
        print(f"  Total Light Curves: {total_optical_count}")
        print(f"  Saved to: {output_h5_path}")

In [ ]:
# Create the dataset with negative GW events
create_dataset_with_neg_gw(
    full_catalog_path=f"{_BASE}/ML+GW+KN/dataset/O5_sim_nsbh_aug/injections_final.csv",
    skymap_dir=f"{_BASE}/data/nsbh_skymap",
    fits_dir=f"{_BASE}/SNANA/SNDATA_ROOT/SIM/LSST_KN_NSBH_AUG",
    output_h5_path=f"{_BASE}/data/LSST_KN_NSBH_AUG/combined_dataset_with_neg_gw.h5",
    pos_indices=pos_indices,
    neg_indices=neg_indices,
    max_neg_gw=None  # Use all negative GW events
)

In [ ]:
# Verify the created dataset
with h5py.File(f"{_BASE}/data/LSST_KN_BNS_AUG/combined_dataset_with_neg_gw.h5", 'r') as f:
    print("Dataset structure:")
    print(f"  n_pos_gw: {f.attrs['n_pos_gw']}")
    print(f"  n_neg_gw: {f.attrs['n_neg_gw']}")
    print(f"  n_total_gw: {f.attrs['n_total_gw']}")
    print(f"  n_total_optical: {f.attrs['n_total_optical']}")
    
    print("\nGW Data:")
    print(f"  scalars shape: {f['events/gw_data/scalars'].shape}")
    print(f"  skymaps shape: {f['events/gw_data/skymaps'].shape}")
    print(f"  has_kn shape: {f['events/gw_data/has_kn'].shape}")
    
    has_kn = f['events/gw_data/has_kn'][:]
    print(f"\n  has_kn=1 (positive): {np.sum(has_kn == 1)}")
    print(f"  has_kn=0 (negative): {np.sum(has_kn == 0)}")
    
    print("\nOptical Data:")
    print(f"  values shape: {f['events/optical_data/values'].shape}")
    print(f"  parent_gw_idx shape: {f['events/optical_data/parent_gw_idx'].shape}")
    
    # Verify parent indices only point to positive GW
    parent_indices = f['events/optical_data/parent_gw_idx'][:]
    print(f"\n  Parent GW index range: [{parent_indices.min()}, {parent_indices.max()}]")
    print(f"  All parent indices point to positive GW: {np.all(parent_indices < f.attrs['n_pos_gw'])}")

## Create dataset for negative samples

In [ ]:
# Latest negative-sample parsing utilities (synced with optical_only/create_optical_only_datasets.py)
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import re

BAND_TO_INDEX = {
    'LSST-u': 0,
    'LSST-g': 1,
    'LSST-r': 2,
    'LSST-i': 3,
    'LSST-z': 4,
    'LSST-Y': 5,
    'u': 0,
    'g': 1,
    'r': 2,
    'i': 3,
    'z': 4,
    'Y': 5,
}


def band_index(band_raw):
    band = str(band_raw).strip()
    if band in BAND_TO_INDEX:
        return BAND_TO_INDEX[band]
    if band.startswith('LSST-'):
        tail = band.split('-', 1)[1]
        return BAND_TO_INDEX.get(tail)
    return None


def first_detection_index(flux, fluxerr, photflag=None, snr_threshold=5.0):
    """Return index of first detection: PHOTFLAG!=0, else SNR threshold fallback."""
    if flux.size == 0:
        return None, None

    if photflag is not None:
        det_mask = np.asarray(photflag, dtype=np.int64) != 0
        if np.any(det_mask):
            return int(np.argmax(det_mask)), 'photflag'

    valid = np.isfinite(fluxerr) & (fluxerr > 0)
    if np.any(valid):
        snr = np.full(flux.shape, -np.inf, dtype=np.float64)
        snr[valid] = flux[valid] / fluxerr[valid]
        det_mask = snr > float(snr_threshold)
        if np.any(det_mask):
            return int(np.argmax(det_mask)), 'snr'

    return None, None


def normalize_flux(flux, fluxerr):
    std = float(np.std(flux))
    mean = float(np.mean(flux))
    scale = std + 1e-8
    return (flux - mean) / scale, fluxerr / scale


def format_realization(mjd, flux, fluxerr, flt, ra, dec, t0_mjd):
    rel_times = (mjd - float(t0_mjd)) / 100.0

    if len(mjd) > MAX_LC_LENGTH:
        keep = np.argsort(np.abs(rel_times))[:MAX_LC_LENGTH]
        keep = np.sort(keep)
        rel_times = rel_times[keep]
        flux = flux[keep]
        fluxerr = fluxerr[keep]
        flt = flt[keep]

    seq_len = min(len(rel_times), MAX_LC_LENGTH)
    val_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)
    err_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)
    mask_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)
    time_vec = np.zeros((MAX_LC_LENGTH,), dtype=np.float32)

    for t in range(seq_len):
        b_idx = band_index(flt[t])
        if b_idx is None:
            continue
        val_mat[t, b_idx] = flux[t]
        err_mat[t, b_idx] = fluxerr[t]
        mask_mat[t, b_idx] = 1.0
        time_vec[t] = rel_times[t]

    if not np.any(mask_mat > 0):
        raise ValueError('realization has no valid band after formatting')

    coords = np.array([ra, dec], dtype=np.float32)
    return val_mat, err_mat, mask_mat, time_vec, coords, float(t0_mjd)


def infer_transient_type(folder_name):
    name = folder_name.lower()
    if 'kn' in name:
        return None
    if 'agn' in name:
        return 'AGN'
    if 'tde' in name:
        return 'TDE'
    if 'ulens' in name:
        return 'uLens'
    if 'dwarf-nova' in name:
        return 'dwarf-nova'
    if 'sn' in name or 'slsn' in name or 'pisn' in name:
        return 'SN'
    return None


def iter_negative_head_files(sim_root):
    sim_root = Path(sim_root)
    files = list(sim_root.rglob('*_HEAD.FITS')) + list(sim_root.rglob('*_HEAD.FITS.gz'))
    return sorted(set(files))


def find_pair_phot_file(head_path):
    head_path = Path(head_path)
    name = head_path.name
    if name.endswith('_HEAD.FITS'):
        p = head_path.with_name(name.replace('_HEAD.FITS', '_PHOT.FITS'))
        return p if p.exists() else None
    if name.endswith('_HEAD.FITS.gz'):
        p = head_path.with_name(name.replace('_HEAD.FITS.gz', '_PHOT.FITS.gz'))
        return p if p.exists() else None
    return None


def parse_negative_file(
    head_path,
    phot_path,
    transient_type,
    snr_threshold=5.0,
    min_nobs=5,
    fixed_offset_days=0.0,
):
    """Parse one HEAD/PHOT pair and return formatted negative light curves + stats."""
    stats = {
        'n_realizations_total': 0,
        'n_realizations_kept': 0,
        'drop_nobs': 0,
        'drop_no_detection': 0,
        'drop_empty_or_invalid': 0,
    }
    out = []

    try:
        with fits.open(head_path, memmap=False) as hdul_head, fits.open(phot_path, memmap=False) as hdul_phot:
            data_head = hdul_head[1].data
            data_phot = hdul_phot[1].data

            ptrobs_min = data_head['PTROBS_MIN']
            ptrobs_max = data_head['PTROBS_MAX']
            mjd_all = data_phot['MJD']
            flux_all = data_phot['FLUXCAL']
            fluxerr_all = data_phot['FLUXCALERR']
            flt_all = data_phot['BAND']
            photflag_all = data_phot['PHOTFLAG'] if 'PHOTFLAG' in data_phot.columns.names else None

            stats['n_realizations_total'] = int(len(data_head))
            for i in range(len(data_head)):
                nobs = int(data_head['NOBS'][i])
                if nobs < int(min_nobs):
                    stats['drop_nobs'] += 1
                    continue

                start_idx = int(ptrobs_min[i]) - 1
                end_idx = int(ptrobs_max[i])
                if start_idx < 0 or end_idx <= start_idx or end_idx > len(mjd_all):
                    stats['drop_empty_or_invalid'] += 1
                    continue

                lc_mjd = np.asarray(mjd_all[start_idx:end_idx], dtype=np.float64)
                lc_flux = np.asarray(flux_all[start_idx:end_idx], dtype=np.float64)
                lc_fluxerr = np.asarray(fluxerr_all[start_idx:end_idx], dtype=np.float64)
                lc_flt = np.asarray(flt_all[start_idx:end_idx])
                lc_photflag = (
                    np.asarray(photflag_all[start_idx:end_idx], dtype=np.int64)
                    if photflag_all is not None
                    else None
                )

                if lc_mjd.size == 0:
                    stats['drop_empty_or_invalid'] += 1
                    continue

                det_idx, _ = first_detection_index(
                    flux=lc_flux,
                    fluxerr=lc_fluxerr,
                    photflag=lc_photflag,
                    snr_threshold=snr_threshold,
                )
                if det_idx is None:
                    stats['drop_no_detection'] += 1
                    continue

                t0_mjd = float(lc_mjd[det_idx]) + float(fixed_offset_days)
                lc_flux, lc_fluxerr = normalize_flux(lc_flux, lc_fluxerr)

                try:
                    ra = float(data_head['RA'][i])
                    dec = float(data_head['DEC'][i])
                    out.append(
                        format_realization(
                            mjd=lc_mjd,
                            flux=lc_flux,
                            fluxerr=lc_fluxerr,
                            flt=lc_flt,
                            ra=ra,
                            dec=dec,
                            t0_mjd=t0_mjd,
                        )
                    )
                    stats['n_realizations_kept'] += 1
                except Exception:
                    stats['drop_empty_or_invalid'] += 1
                    continue
    except Exception as e:
        print(f'Error processing FITS for {head_path}: {e}')
        return out, stats

    return out, stats


In [ ]:
neg_lcs, neg_stats = parse_negative_file(
    head_path=f"{_BASE}/data/Tutorial_LSST_sims_2025/sims/SNIa/AMR_ELASTICC2_LSST_NONIaMODEL00-0012_HEAD.FITS",
    phot_path=f"{_BASE}/data/Tutorial_LSST_sims_2025/sims/SNIa/AMR_ELASTICC2_LSST_NONIaMODEL00-0012_PHOT.FITS",
    transient_type='SN',
    snr_threshold=5.0,
    min_nobs=5,
    fixed_offset_days=0.0,
)
print(f'parsed light curves: {len(neg_lcs)}')
print(neg_stats)


In [ ]:
# Create negative HDF5 dataset (uniform cls-time anchor range, no GW H5 required)
def _parse_negative_file_worker(
    head_path_str,
    phot_path_str,
    transient_type,
    snr_threshold,
    min_nobs,
    fixed_offset_days,
):
    lcs, stats = parse_negative_file(
        head_path=Path(head_path_str),
        phot_path=Path(phot_path_str),
        transient_type=transient_type,
        snr_threshold=snr_threshold,
        min_nobs=min_nobs,
        fixed_offset_days=fixed_offset_days,
    )
    return transient_type, lcs, stats


def _parallel_map_with_fallback(
    fn,
    arg_rows,
    worker_count=1,
    chunksize=1,
    total=0,
    desc='',
):
    if len(arg_rows) == 0:
        return

    cols = list(zip(*arg_rows))
    iterables = tuple([list(col) for col in cols])

    try:
        with ProcessPoolExecutor(max_workers=worker_count) as executor:
            mapped = executor.map(fn, *iterables, chunksize=chunksize)
            for item in tqdm(mapped, total=total, desc=desc):
                yield item
    except (PermissionError, OSError, RuntimeError) as exc:
        print(
            f"[WARN] ProcessPool unavailable ({exc}); fallback to ThreadPoolExecutor with {worker_count} workers."
        )
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            mapped = executor.map(fn, *iterables)
            for item in tqdm(mapped, total=total, desc=desc):
                yield item


def _create_optical_group(grp, chunk_size, create_cls_base=True):
    ds_values = grp.create_dataset(
        'values',
        (0, MAX_LC_LENGTH, NUM_BANDS),
        maxshape=(None, MAX_LC_LENGTH, NUM_BANDS),
        dtype='f4',
        chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS),
    )
    ds_errors = grp.create_dataset(
        'errors',
        (0, MAX_LC_LENGTH, NUM_BANDS),
        maxshape=(None, MAX_LC_LENGTH, NUM_BANDS),
        dtype='f4',
        chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS),
    )
    ds_masks = grp.create_dataset(
        'masks',
        (0, MAX_LC_LENGTH, NUM_BANDS),
        maxshape=(None, MAX_LC_LENGTH, NUM_BANDS),
        dtype='f4',
        chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS),
    )
    ds_times = grp.create_dataset(
        'times',
        (0, MAX_LC_LENGTH),
        maxshape=(None, MAX_LC_LENGTH),
        dtype='f4',
        chunks=(chunk_size, MAX_LC_LENGTH),
    )
    ds_zero_time_mjd_base = grp.create_dataset(
        'zero_time_mjd_base',
        (0,),
        maxshape=(None,),
        dtype='f8',
        chunks=(chunk_size,),
    )
    ds_zero_time_mjd_cls_base = None
    if create_cls_base:
        ds_zero_time_mjd_cls_base = grp.create_dataset(
            'zero_time_mjd_cls_base',
            (0,),
            maxshape=(None,),
            dtype='f8',
            chunks=(chunk_size,),
        )
    ds_coords = grp.create_dataset(
        'coordinates',
        (0, 2),
        maxshape=(None, 2),
        dtype='f4',
        chunks=(chunk_size, 2),
    )
    return ds_values, ds_errors, ds_masks, ds_times, ds_zero_time_mjd_base, ds_zero_time_mjd_cls_base, ds_coords


def create_negative_dataset(
    sim_root,
    output_h5_path,
    neg_group='negative/optical_data',
    snr_threshold=5.0,
    min_nobs=5,
    fixed_offset_days=0.0,
    max_negative_heads=None,
    buffer_limit=5000,
    num_workers=1,
    cls_time_anchor_mjd_min=None,
    cls_time_anchor_mjd_max=None,
    cls_time_anchor_seed=42,
):
    """
    Build negative optical dataset using first-detection anchored time zero.

    zero_time_mjd_cls_base is sampled uniformly from
    [cls_time_anchor_mjd_min, cls_time_anchor_mjd_max].
    """
    sim_root = Path(sim_root)
    output_h5_path = Path(output_h5_path)
    output_h5_path.parent.mkdir(parents=True, exist_ok=True)

    if cls_time_anchor_mjd_min is None or cls_time_anchor_mjd_max is None:
        raise ValueError('cls_time_anchor_mjd_min/max are required for uniform cls-time sampling.')
    cls_time_anchor_mjd_min = float(cls_time_anchor_mjd_min)
    cls_time_anchor_mjd_max = float(cls_time_anchor_mjd_max)
    if not np.isfinite(cls_time_anchor_mjd_min) or not np.isfinite(cls_time_anchor_mjd_max):
        raise ValueError('cls_time_anchor_mjd_min/max must be finite numbers.')
    if cls_time_anchor_mjd_max <= cls_time_anchor_mjd_min:
        raise ValueError('cls_time_anchor_mjd_max must be greater than cls_time_anchor_mjd_min.')
    rng_anchor = np.random.default_rng(int(cls_time_anchor_seed))

    chunk_size = 1024
    dt_str = h5py.string_dtype(encoding='utf-8')

    with h5py.File(output_h5_path, 'w') as f:
        grp = f.create_group(neg_group)
        (
            ds_values,
            ds_errors,
            ds_masks,
            ds_times,
            ds_zero_time_mjd_base,
            ds_zero_time_mjd_cls_base,
            ds_coords,
        ) = _create_optical_group(grp, chunk_size, create_cls_base=True)
        ds_types = grp.create_dataset('types', (0,), maxshape=(None,), dtype=dt_str, chunks=(chunk_size,))

        total_optical = 0
        b_vals = []
        b_errs = []
        b_masks = []
        b_times = []
        b_zero_time_mjd_base = []
        b_zero_time_mjd_cls_base = []
        b_coords = []
        b_types = []

        stats = {
            'head_files_total': 0,
            'head_files_used': 0,
            'drop_nobs': 0,
            'drop_no_detection': 0,
            'drop_empty_or_invalid': 0,
        }

        def flush():
            nonlocal total_optical
            if not b_vals:
                return
            n_new = len(b_vals)
            cur = total_optical
            new_size = cur + n_new

            ds_values.resize(new_size, axis=0)
            ds_errors.resize(new_size, axis=0)
            ds_masks.resize(new_size, axis=0)
            ds_times.resize(new_size, axis=0)
            ds_zero_time_mjd_base.resize(new_size, axis=0)
            ds_zero_time_mjd_cls_base.resize(new_size, axis=0)
            ds_coords.resize(new_size, axis=0)
            ds_types.resize(new_size, axis=0)

            ds_values[cur:new_size] = np.asarray(b_vals, dtype=np.float32)
            ds_errors[cur:new_size] = np.asarray(b_errs, dtype=np.float32)
            ds_masks[cur:new_size] = np.asarray(b_masks, dtype=np.float32)
            ds_times[cur:new_size] = np.asarray(b_times, dtype=np.float32)
            ds_zero_time_mjd_base[cur:new_size] = np.asarray(b_zero_time_mjd_base, dtype=np.float64)
            ds_zero_time_mjd_cls_base[cur:new_size] = np.asarray(b_zero_time_mjd_cls_base, dtype=np.float64)
            ds_coords[cur:new_size] = np.asarray(b_coords, dtype=np.float32)
            ds_types[cur:new_size] = np.asarray(b_types, dtype=object)

            total_optical = new_size
            b_vals.clear()
            b_errs.clear()
            b_masks.clear()
            b_times.clear()
            b_zero_time_mjd_base.clear()
            b_zero_time_mjd_cls_base.clear()
            b_coords.clear()
            b_types.clear()

        worker_count = max(1, int(num_workers))

        def consume_negative_result(transient_type, lcs, file_stats):
            stats['head_files_used'] += 1
            stats['drop_nobs'] += int(file_stats['drop_nobs'])
            stats['drop_no_detection'] += int(file_stats['drop_no_detection'])
            stats['drop_empty_or_invalid'] += int(file_stats['drop_empty_or_invalid'])

            for vals, errs, masks, times, coords, zero_time_mjd_base in lcs:
                b_vals.append(vals)
                b_errs.append(errs)
                b_masks.append(masks)
                b_times.append(times)
                z_base = float(zero_time_mjd_base)
                z_cls = float(rng_anchor.uniform(cls_time_anchor_mjd_min, cls_time_anchor_mjd_max))
                b_zero_time_mjd_base.append(z_base)
                b_zero_time_mjd_cls_base.append(z_cls)
                b_coords.append(coords)
                b_types.append(transient_type)

            if len(b_vals) >= int(buffer_limit):
                flush()

        head_files = iter_negative_head_files(sim_root)
        if max_negative_heads is not None:
            head_files = head_files[: int(max_negative_heads)]
        stats['head_files_total'] = int(len(head_files))

        tasks = []
        for head_path in head_files:
            transient_type = infer_transient_type(head_path.parent.name)
            if transient_type is None:
                continue
            phot_path = find_pair_phot_file(head_path)
            if phot_path is None:
                continue
            tasks.append((str(head_path), str(phot_path), transient_type))

        if worker_count == 1 or len(tasks) <= 1:
            for head_path_str, phot_path_str, transient_type in tqdm(tasks, desc='Negative HEAD files'):
                lcs, file_stats = parse_negative_file(
                    head_path=Path(head_path_str),
                    phot_path=Path(phot_path_str),
                    transient_type=transient_type,
                    snr_threshold=float(snr_threshold),
                    min_nobs=int(min_nobs),
                    fixed_offset_days=float(fixed_offset_days),
                )
                consume_negative_result(transient_type, lcs, file_stats)
        else:
            n_rows = len(tasks)
            chunksize = max(1, n_rows // (worker_count * 8))
            arg_rows = [
                (
                    tasks[i][0],
                    tasks[i][1],
                    tasks[i][2],
                    float(snr_threshold),
                    int(min_nobs),
                    float(fixed_offset_days),
                )
                for i in range(n_rows)
            ]
            for transient_type, lcs, file_stats in _parallel_map_with_fallback(
                _parse_negative_file_worker,
                arg_rows=arg_rows,
                worker_count=worker_count,
                chunksize=chunksize,
                total=n_rows,
                desc='Negative HEAD files',
            ):
                consume_negative_result(transient_type, lcs, file_stats)

        flush()

        f.attrs['n_total_optical'] = int(total_optical)
        for k, v in stats.items():
            f.attrs[k] = int(v)
        f.attrs['time_zero_anchor'] = 'first_detection'
        f.attrs['first_detection_rule'] = 'photflag_nonzero_else_snr_gt_5'
        f.attrs['time_scale_divisor_days'] = 100.0
        f.attrs['time_zero_version'] = 'fd_v1'
        f.attrs['drop_no_detection_count'] = int(stats['drop_no_detection'])
        f.attrs['snr_threshold'] = float(snr_threshold)
        f.attrs['detection_photflags'] = 'nonzero'
        f.attrs['fixed_offset_days'] = float(fixed_offset_days)
        f.attrs['num_workers'] = int(worker_count)
        f.attrs['time_zero_base_semantics'] = 'first_detection_mjd_plus_fixed_offset_days'
        f.attrs['time_unit'] = 'mjd_days'
        f.attrs['runtime_offset_applied'] = 1
        f.attrs['cls_anchor_policy'] = 'uniform_range'
        f.attrs['cls_anchor_mjd_min'] = float(cls_time_anchor_mjd_min)
        f.attrs['cls_anchor_mjd_max'] = float(cls_time_anchor_mjd_max)
        f.attrs['cls_anchor_seed'] = int(cls_time_anchor_seed)

    print(f'Total negative light curves: {total_optical}')
    print(f'Saved to: {output_h5_path}')


In [ ]:
# # ELASTICC1
# create_negative_dataset(
#     sim_root=f'{_BASE}/data/ELASTICC_TRAIN',
#     output_h5_path=f'{_BASE}/data/ELASTICC_TRAIN/ELASTICC_negative_dataset.h5',
#     neg_group='ELASTICC/optical_data',
#     snr_threshold=5.0,
#     min_nobs=5,
#     fixed_offset_days=0.0,
#     cls_time_anchor_mjd_min=60000.0,
#     cls_time_anchor_mjd_max=62000.0,
#     cls_time_anchor_seed=42,
#     num_workers=1,
# )

# ELASTICC2
create_negative_dataset(
    sim_root=f'{_BASE}/data/ELASTICC2_TRAIN_02',
    output_h5_path=f'{_BASE}/data/Optical_Negative_dataset/ELASTICC2_negative_dataset.h5',
    neg_group='ELASTICC2/optical_data',
    snr_threshold=5.0,
    min_nobs=5,
    fixed_offset_days=0.0,
    cls_time_anchor_mjd_min=60000.0,
    cls_time_anchor_mjd_max=62000.0,
    cls_time_anchor_seed=42,
    num_workers=1,
)

# # Tutorial LSST sims
# create_negative_dataset(
#     sim_root=f'{_BASE}/data/Tutorial_LSST_sims_2025/sims/',
#     output_h5_path=f'{_BASE}/data/Tutorial_LSST_sims_2025/Tutorial_negative_dataset.h5',
#     neg_group='Tutorial/optical_data',
#     snr_threshold=5.0,
#     min_nobs=5,
#     fixed_offset_days=0.0,
#     cls_time_anchor_mjd_min=60000.0,
#     cls_time_anchor_mjd_max=62000.0,
#     cls_time_anchor_seed=42,
#     num_workers=1,
# )


## Functions for make dataloader

In [ ]:
from collections import defaultdict
from typing import List, Iterator
from torch.utils.data import Dataset, DataLoader, Sampler

In [ ]:
def build_gw_to_lc_mapping(h5_path: str):
    """
    Scans the HDF5 file to build a mapping from GW Event Index to Light Curve Indices.
    This is required for the Balanced Sampler.
    
    Args:
        h5_path: Path to the HDF5 file.
        
    Returns:
        gw_to_lc_map: Dictionary {gw_idx: np.array([lc_idx_1, lc_idx_2, ...])}
    """
    print(f"Building GW-to-Optical index mapping from {h5_path}...")
    with h5py.File(h5_path, 'r') as f:
        # Load the parent_gw_idx array into memory (it's essentially a list of integers)
        # Shape: [Total_Optical_Samples]
        all_parent_indices = f['events/optical_data/parent_gw_idx'][:]
        
    gw_to_lc_map = defaultdict(list)
    for lc_idx, gw_idx in enumerate(all_parent_indices):
        gw_to_lc_map[gw_idx].append(lc_idx)
        
    # Convert lists to numpy arrays for faster random sampling later
    final_map = {k: np.array(v) for k, v in gw_to_lc_map.items()}
    
    print(f"Mapping complete. Found {len(final_map)} unique GW events.")
    return final_map

In [ ]:
gw_to_lc_map = build_gw_to_lc_mapping(f"{_BASE}/data/LSST_KN_BNS/combined_dataset.h5")
gw_to_lc_map[0]

In [ ]:
class RelationalHDF5Dataset(Dataset):
    """
    PyTorch Dataset for the relational HDF5 structure.
    Reads optical data by index and fetches the corresponding unique GW data.
    """
    def __init__(self, h5_path: str):
        super().__init__()
        self.h5_path = h5_path
        self.h5_file = None
        
        # Open file temporarily to get dataset length
        with h5py.File(h5_path, 'r') as f:
            self.length = f['events/optical_data/values'].shape[0]
            
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        """
        Args:
            idx: Index of the light curve (optical data).
        """
        # Lazy loading: Open file only when needed (crucial for num_workers > 0)
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
            
        # 1. Retrieve Optical Data (Values, Errors, Masks, Times)
        #    HDF5 structure: events/optical_data/...
        opt_val   = torch.from_numpy(self.h5_file['events/optical_data/values'][idx])
        opt_err   = torch.from_numpy(self.h5_file['events/optical_data/errors'][idx])
        opt_mask  = torch.from_numpy(self.h5_file['events/optical_data/masks'][idx])
        opt_time  = torch.from_numpy(self.h5_file['events/optical_data/times'][idx])
        opt_coords = torch.from_numpy(self.h5_file['events/optical_data/coordinates'][idx])
        
        # 2. Retrieve Parent GW Index
        gw_idx = self.h5_file['events/optical_data/parent_gw_idx'][idx]
        
        # 3. Retrieve Unique GW Data using gw_idx
        #    HDF5 structure: events/gw/...
        gw_scalar = torch.from_numpy(self.h5_file['events/gw_data/scalars'][gw_idx])
        gw_skymap = torch.from_numpy(self.h5_file['events/gw_data/skymaps'][gw_idx])
        
        # Return tuple: (GW_Inputs, Optical_Inputs, Metadata)
        # gw_idx is returned for masking the contrastive loss (handling same-source negatives)
        return gw_scalar, gw_skymap, opt_time, opt_val, opt_mask, opt_err, opt_coords, int(gw_idx)
    
class BalancedGWBatchedSampler(Sampler):
    """
    Custom Batch Sampler that ensures:
    1. Each batch contains 'batch_size' UNIQUE GW events.
    2. For each selected GW event, ONE light curve is randomly sampled.
    
    This prevents "false negatives" in contrastive learning where multiple LCs 
    from the same GW event appear in the same batch.
    """
    def __init__(self, gw_to_lc_map: dict, batch_size: int, steps_per_epoch: int):
        """
        Args:
            gw_to_lc_map: Dictionary mapping GW_ID -> [LC_ID_1, LC_ID_2, ...]
            batch_size: Number of unique GW events per batch.
            steps_per_epoch: Number of batches to yield per 'epoch'. (N_LC // batch_size)
        """
        self.gw_to_lc_map = gw_to_lc_map
        self.unique_gw_ids = list(gw_to_lc_map.keys())
        self.batch_size = batch_size
        self.steps_per_epoch = steps_per_epoch
        
        # Validation: Batch size cannot exceed total unique GW events
        if self.batch_size > len(self.unique_gw_ids):
            raise ValueError(f"Batch size ({batch_size}) > Unique GW events ({len(self.unique_gw_ids)}).")

    def __iter__(self) -> Iterator[List[int]]:
        for _ in range(self.steps_per_epoch):
            # 1. Sample unique GW IDs for this batch (without replacement)
            batch_gw_ids = np.random.choice(
                self.unique_gw_ids, 
                size=self.batch_size, 
                replace=False
            )
            
            batch_lc_indices = []
            
            # 2. For each GW ID, sample ONE random light curve index
            for gw_id in batch_gw_ids:
                possible_lcs = self.gw_to_lc_map[gw_id]
                chosen_lc = np.random.choice(possible_lcs)
                batch_lc_indices.append(chosen_lc)
                
            # Yield the list of optical indices for the DataLoader to fetch
            yield batch_lc_indices

    def __len__(self):
        return self.steps_per_epoch

def create_training_dataloader(
    h5_path: str, 
    batch_size: int = 32, 
    steps_per_epoch: int = 1000, 
    num_workers: int = 4
):
    """
    Factory function to initialize the Dataset, Sampler, and DataLoader.
    """
    # 1. Build Index Map (Once)
    gw_map = build_gw_to_lc_mapping(h5_path)
    
    # 2. Initialize Dataset
    dataset = RelationalHDF5Dataset(h5_path)
    
    # 3. Initialize Custom Sampler
    # Note: 'steps_per_epoch' defines how many batches constitute one epoch loop
    sampler = BalancedGWBatchedSampler(
        gw_to_lc_map=gw_map,
        batch_size=batch_size,
        steps_per_epoch=steps_per_epoch
    )
    
    # 4. Initialize DataLoader
    # IMPORTANT: batch_sampler is used, so batch_size/shuffle/sampler/drop_last 
    # arguments in DataLoader constructor must not be provided.
    loader = DataLoader(
        dataset,
        batch_sampler=sampler,
        num_workers=num_workers,
        pin_memory=True
    )
    
    return loader

In [ ]:
h5file = f"{_BASE}/data/LSST_KN_BNS/combined_dataset.h5"
dataloader = create_training_dataloader(
    h5_path=h5file,
    batch_size=32,
    steps_per_epoch=10000
)
print("\nTesting DataLoader...")
for batch_idx, batch_data in enumerate(dataloader):
    # Unpack data
    # Order matches __getitem__: scalar, skymap, time, val, mask, err, gw_idx
    gw_s, gw_m, opt_t, opt_v, opt_mask, opt_err, opt_coords, gw_indices = batch_data
    
    print(f"Batch {batch_idx}:")
    print(f"  GW Scalars: {gw_s.shape}")    # Expected: [B, 7]
    print(f"  GW Skymap:  {gw_m.shape}")    # Expected: [B, 6, 19200]
    print(f"  Opt Values: {opt_v.shape}")   # Expected: [B, 200, 6]
    print(f"  Opt Errors: {opt_err.shape}") # Expected: [B, 200, 6]
    print(f"  Opt Masks:  {opt_mask.shape}")# Expected: [B, 200, 6]
    print(f"  Opt Times:  {opt_t.shape}")   # Expected: [B, 200]
    print(f"  Opt Coords: {opt_coords.shape}") # Expected: [B, 2]
    print(f"  GW Indices:{gw_indices.shape}") # Expected: [B]
    
    # Verify Unique GWs
    unique_gws = torch.unique(gw_indices)
    print(f"  Unique GWs in batch: {len(unique_gws)} (Should be Batch Size: {gw_s.shape[0]})")
    
    break